# Retrain T1.5 + H-R9 corpus balance ablation

After H-R3 (hard-neg margin filter) ruled itself out on
2026-04-19 (arm_a regressed macro -2.49pp), the remaining
suspects for the NFCorpus gap vs v5 are **corpus balance** and
**training volume**. Both are pure hyperparameter knobs, no code
changes required.

Three arms, same stores + eval set + seed as the 2026-04-19
baseline (SciFact 0.7786 / NFCorpus 0.3677 / FiQA 0.4568 final
absolute NDCG@10, macro +4.14%):

| Arm | `temperature` | `total_triples` | Isolates |
|---|---|---|---|
| arm_t03     | `0.3` | `30000` | corpus balance (more NFCorpus share, same volume) |
| arm_uniform | uniform strategy | `30000` | maximum balance lever (NFCorpus gets 33%) |
| arm_vol     | `0.5` | `60000` | training volume (same balance as baseline, 2x volume) |

Target: NFCorpus final absolute NDCG@10 > 0.38 on at least one
arm, without regressing SciFact (stay above 0.775) or FiQA
(stay above 0.45).

Runtime: arm_t03 + arm_uniform ~30 min each on T4 (same total
triples as baseline), arm_vol ~55 min (double the triples).
Total ~2h. Run selectively; the summary cell at the end auto-skips
arms that never ran.

In [ ]:
# Cell 1: Setup -- clone develop (H-R9 is pure CLI, no branch needed).
!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0'
!rm -rf /content/vstash
!git clone --branch develop https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .

In [ ]:
# Cell 2: Download BEIR + ingest into per-dataset stores.
import os
import shutil
import sys

os.chdir("/content/vstash")
sys.path.insert(0, "/content/vstash")
os.makedirs("experiments/data", exist_ok=True)

from experiments.beir_benchmark import download_beir, load_beir
from sentence_transformers import SentenceTransformer
from vstash.store import VstashStore

BASE_MODEL = "BAAI/bge-small-en-v1.5"
DATASETS = ["scifact", "nfcorpus", "fiqa"]
store_paths = {name: f"/tmp/retrain_hr9_{name}.db" for name in DATASETS}

for p in store_paths.values():
    for suffix in ("", "-wal", "-shm"):
        target = p + suffix
        if os.path.isfile(target):
            os.remove(target)

model = SentenceTransformer(BASE_MODEL)

per_dataset = {}
stores = {}
for name in DATASETS:
    cache = download_beir(name)
    corpus, queries, qrels = load_beir(cache)
    doc_ids = list(corpus.keys())
    print(f"[{name}] corpus: {len(doc_ids)} docs | queries: {len(queries)} | qrels: {len(qrels)}")
    per_dataset[name] = {"corpus": corpus, "queries": queries, "qrels": qrels}

    texts = [
        (corpus[d].get("title", "") + " " + corpus[d].get("text", "")).strip() for d in doc_ids
    ]
    vecs = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=128)
    store = VstashStore(store_paths[name], embedding_dim=int(vecs.shape[1]))
    # TODO: batch via store.add_documents_batch when ingest time
    # becomes a bottleneck. On the 57k-chunk FiQA ingest this
    # per-doc loop takes ~30 s; acceptable for an ablation
    # that runs once per notebook session.
    for doc_id, text, vec in zip(doc_ids, texts, vecs):
        store.add_document(
            path=f"{name}://{doc_id}",
            title=corpus[doc_id].get("title", "")[:80] or doc_id,
            chunks=[text],
            embeddings=[list(map(float, vec))],
        )
    stats = store.stats()
    print(f"  -> ingested {stats.documents} docs / {stats.chunks} chunks")
    stores[name] = store

In [ ]:
# Cell 3: Build per-dataset eval sets from real qrels. Reused as
# training_queries_by_dataset too (T1.5 v5 recipe).
from vstash.retrain import qrels_to_eval_queries

eval_queries_by_dataset = {}
for name, bundle in per_dataset.items():
    eqs = qrels_to_eval_queries(
        queries=bundle["queries"],
        qrels=bundle["qrels"],
        path_for_doc_id=lambda doc_id, d=name: f"{d}://{doc_id}",
    )
    eval_queries_by_dataset[name] = eqs
    print(f"[{name}] eval_queries (real qrels): {len(eqs)}")

In [ ]:
# Cell 4: Shared retrain_multi config. Arm cells below only override
# sampling + temperature + total_triples + output_path.
import time
from vstash.retrain import retrain_multi

EVAL_NOISE = max(max(s.stats().chunks for s in stores.values()), 10000)
SEED = 42


def run_arm(
    arm_name: str,
    sampling: str,
    temperature: float,
    total_triples: int,
):
    output_path = f"/content/retrained_hr9_{arm_name}"
    for suffix in ("", ".candidate", ".old"):
        p = output_path + suffix
        if os.path.isdir(p):
            shutil.rmtree(p)
    t0 = time.perf_counter()
    result = retrain_multi(
        stores,
        base_model=BASE_MODEL,
        output_path=output_path,
        sampling=sampling,
        temperature=temperature,
        total_triples=total_triples,
        epochs=2,
        lr=3e-6,
        batch_size=32,
        use_amp=True,
        max_seq_length=256,
        bulk_mine=True,
        bulk_eval=True,
        # Note on query overlap: we pass the same BEIR qrels as
        # both training and eval queries to reproduce the v5
        # recipe that produced bge-small-rrf-v2. This does
        # overlap the test signal, and ABSOLUTE BEIR deltas
        # published this way are not apples-to-apples with
        # methods (e.g. ColBERTv2) that never saw these
        # queries during training. For a pure ablation of
        # corpus balance vs volume, the leakage is held
        # constant across arms and the RELATIVE comparison
        # below stays valid. See the methodological note in
        # experiments/hypotheses.md.
        training_queries_by_dataset=eval_queries_by_dataset,
        eval_queries_by_dataset=eval_queries_by_dataset,
        eval_noise_size=EVAL_NOISE,
        min_gain=-1.0,
        per_dataset_gate=False,
        seed=SEED,
    )
    elapsed = time.perf_counter() - t0
    print(
        f"\n[{arm_name}] sampling={sampling} temp={temperature} total={total_triples}  "
        f"macro delta NDCG@10={result.macro_delta_ndcg * 100:+.2f}%  "
        f"total_pairs={result.total_pairs}  elapsed={elapsed:.1f}s"
    )
    print("Per-dataset final absolute NDCG@10:")
    for ds in DATASETS:
        f = result.per_dataset_final.get(ds)
        if f is not None:
            print(f"  {ds:<10} final={f.ndcg_at_10:.4f}  n={f.n_queries}")
    return result, output_path

## Arms

Each arm is an independent `retrain_multi` call. `min_gain=-1` so
every arm saves its training_meta.json even if the candidate
underperforms. Stores and eval queries are reused across arms.

In [ ]:
# arm_t03: temperature=0.3, total_triples=30000. More NFCorpus share,
# same volume as the T1.5 baseline.
result_t03, path_t03 = run_arm(
    "arm_t03", sampling="temperature", temperature=0.3, total_triples=30000
)

In [ ]:
# arm_uniform: NFCorpus gets 33% (1/N). Upper bound on the balance
# lever; tells us whether NFCorpus can close the gap at all with
# more share of the same-sized budget.
result_uniform, path_uniform = run_arm(
    "arm_uniform", sampling="uniform", temperature=0.0, total_triples=30000
)

In [ ]:
# arm_vol: temperature=0.5 (same as baseline), total=60000.
# Isolates training volume from balance. If this wins vs baseline
# but arm_t03/uniform do not, the diagnosis is 'more pairs', not
# 'more NFCorpus pairs'.
result_vol, path_vol = run_arm(
    "arm_vol", sampling="temperature", temperature=0.5, total_triples=60000
)

In [ ]:
# Summary cell: read every completed arm's training_meta.json and
# build a comparison table of FINAL ABSOLUTE NDCG@10 per dataset +
# macro delta. Remember: judge by final absolute, not delta% (eval
# pipeline caveat, methodological note in experiments/hypotheses.md).
import json
from pathlib import Path

arms = [
    ("baseline*", None, None, None, None),  # reference row (2026-04-19 post-#243)
    ("arm_t03", path_t03 if "path_t03" in dir() else None, "temperature", 0.3, 30000),
    ("arm_uniform", path_uniform if "path_uniform" in dir() else None, "uniform", 0.0, 30000),
    ("arm_vol", path_vol if "path_vol" in dir() else None, "temperature", 0.5, 60000),
]

# Baseline numbers from the 2026-04-19 post-#243 rerun (this is the
# apples-to-apples reference for these arms because all arms use the
# same widened-top-K eval pipeline).
BASELINE_FINAL = {"scifact": 0.7786, "nfcorpus": 0.3677, "fiqa": 0.4568}
BASELINE_MACRO = 0.5344


def _load(path):
    if not path:
        return None
    for candidate in (Path(path), Path(path + ".candidate"), Path(path + ".old")):
        meta_path = candidate / "training_meta.json"
        if meta_path.exists():
            return json.loads(meta_path.read_text())
    return None


print(
    f"{'arm':<12} {'sampling':<12} {'temp':>5}  {'total':>6}  {'pairs':>7}  "
    f"{'SciFact':>9}  {'NFCorpus':>9}  {'FiQA':>9}  {'macro':>9}"
)
print("-" * 110)
# Baseline reference row.
print(
    f"{'baseline*':<12} {'temperature':<12} {'0.5':>5}  {'30000':>6}  {'28490':>7}  "
    f"{BASELINE_FINAL['scifact']:>9.4f}  {BASELINE_FINAL['nfcorpus']:>9.4f}  "
    f"{BASELINE_FINAL['fiqa']:>9.4f}  {BASELINE_MACRO:>9.4f}"
)
# Arm rows from training_meta.json.
for name, path, sampling, temp, total in arms[1:]:
    meta = _load(path)
    if meta is None:
        continue
    multi = meta.get("multi_eval", {})
    pairs = sum(multi.get("per_dataset_pairs", {}).values())
    fin = multi.get("per_dataset_final", {})
    vals = [fin.get(ds, {}).get("ndcg_at_10") for ds in DATASETS]
    macro = multi.get("macro_final_ndcg")
    if None in vals or macro is None:
        continue
    print(
        f"{name:<12} {sampling:<12} {temp:>5.2f}  {total:>6}  {pairs:>7}  "
        f"{vals[0]:>9.4f}  {vals[1]:>9.4f}  {vals[2]:>9.4f}  {macro:>9.4f}"
    )

print()
print("Target: NFCorpus > 0.38 on at least one arm;")
print("        SciFact stays > 0.775, FiQA stays > 0.45.")